### there are 2 part which was written by AI. they are
1) all part reusing phase 2 logic
2) retained_households list in phase 3.2

### reusing phase 2 logic

In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os

path = kagglehub.dataset_download("frtgnn/dunnhumby-the-complete-journey")

tx = pd.read_csv(os.path.join(path, "transaction_data.csv"))

tx.columns = tx.columns.str.lower()

tx.head()

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [2]:
def build_rfm(tx, cutoff_day):
    data = tx[tx["day"] <= cutoff_day]

    rfm = data.groupby("household_key").agg(
        last_day  = ("day", "max"),
        frequency = ("basket_id", "nunique"),
        monetary  = ("sales_value", "sum"),
    )
    rfm["recency"] = cutoff_day - rfm["last_day"]

    r_conditions = [rfm["recency"] <= 7, rfm["recency"] <= 14,
                    rfm["recency"] <= 30, rfm["recency"] <= 60]
    rfm["r_score"] = np.select(r_conditions, [5, 4, 3, 2], default=1)
    rfm["f_score"] = pd.qcut(rfm["frequency"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    rfm["m_score"] = pd.qcut(rfm["monetary"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    rfm["value_score"] = (rfm["f_score"] + rfm["m_score"]) / 2

    seg_conditions = [
        (rfm["r_score"] >= 4) & (rfm["value_score"] >= 4),
        (rfm["r_score"] >= 4) & (rfm["value_score"] >= 3),
        (rfm["r_score"] <= 3) & (rfm["value_score"] >= 3),
        (rfm["r_score"] <= 2) & (rfm["value_score"] < 3),
    ]
    seg_names = ["High-Value Active", "Mid-Value Active", "High-Value Lapsing", "Low-Value Lapsed"]
    rfm["segments"] = np.select(seg_conditions, seg_names, default="Low-Value Active")

    return rfm

In [3]:
build_rfm(tx, cutoff_day=711)["segments"].value_counts()

segments
High-Value Active     782
Low-Value Active      733
Mid-Value Active      418
Low-Value Lapsed      372
High-Value Lapsing    195
Name: count, dtype: int64

In [4]:
rfm_cutoff = build_rfm(tx, cutoff_day=669)

In [5]:
rfm_cutoff

,last_day,frequency,monetary,recency,r_score,f_score,m_score,value_score,segments
household_key,,,,,,,,,
1,660,79,3959.91,9,4,3,4,3.5,Mid-Value Active
2,668,45,1954.34,1,5,2,3,2.5,Low-Value Active
3,640,45,2594.30,29,3,2,4,3.0,High-Value Lapsing
4,627,30,1200.11,42,2,1,2,1.5,Low-Value Lapsed
5,589,38,749.09,80,1,2,2,2.0,Low-Value Lapsed
...,...,...,...,...,...,...,...,...,...
2496,662,60,4105.29,7,5,3,4,3.5,Mid-Value Active
2497,665,216,6874.67,4,5,5,5,5.0,High-Value Active
2498,666,159,2511.39,3,5,5,3,4.0,High-Value Active


### phase 3.1 – choose the churn window

In [6]:
tx

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2595727,1598,42305362535,711,92130,1,0.99,3228,0.00,1520,102,0.0,0.0
2595728,1598,42305362535,711,114102,1,8.89,3228,0.00,1520,102,0.0,0.0
2595729,1598,42305362535,711,133449,1,6.99,3228,0.00,1520,102,0.0,0.0
2595730,1598,42305362535,711,6923644,1,4.50,3228,-0.49,1520,102,0.0,0.0


In [7]:
tx_unduplicated = tx[["household_key", "day"]].drop_duplicates().sort_values(["household_key", "day"])

In [8]:
tx_unduplicated

,household_key,day
46996,1,51
71711,1,67
123815,1,88
140773,1,94
172266,1,101
...,...,...
2531288,2500,695
2543184,2500,698
2565279,2500,704
2574778,2500,706


In [9]:
gaps = tx_unduplicated.groupby("household_key")["day"].diff().dropna()

In [10]:
gaps

71711      16.0
123815     21.0
140773      6.0
172266      7.0
190170      7.0
           ... 
2531288     4.0
2543184     3.0
2565279     6.0
2574778     2.0
2581430     2.0
Name: day, Length: 223033, dtype: float64

In [11]:
print(gaps.describe())
print(gaps.quantile([0.90, 0.95, 0.98, 0.99]))

count    223033.000000
mean          6.943869
std          15.870607
min           1.000000
25%           2.000000
50%           3.000000
75%           7.000000
max         666.000000
Name: day, dtype: float64
0.90    14.0
0.95    22.0
0.98    40.0
0.99    61.0
Name: day, dtype: float64


In [12]:
(gaps > 42).mean()

np.float64(0.01817668237435716)

In [13]:
(gaps > 60).mean()

np.float64(0.010307891657288383)

Choosing the churn window (42 days)

To avoid an arbitrary threshold, I computed the gap in days between consecutive shopping days for every household (223,033 gaps).

Median gap: 3 days. 95th percentile: 22 days. 99th percentile: 61 days.
Only 1.8% of gaps exceed 42 days; 1.0% exceed 60 days.

A 22-day window (95th percentile) is too short: a typical household makes ~80 visits in two years, so a 5% tail means several normal breaks of 22+ days, and loyal customers on holiday would be labelled as churned. At 42 days, false churn labels drop to under 2% of normal gaps. Extending to 60 days only reduces this by 0.8 points but moves the cutoff 18 days earlier, so 42 days (6 weeks) is the better trade-off.

### phase 3.2 – select active households at the cutoff and create churn labels

In [14]:
rfm_active = rfm_cutoff[rfm_cutoff.recency < 42]
rfm_active

,last_day,frequency,monetary,recency,r_score,f_score,m_score,value_score,segments
household_key,,,,,,,,,
1,660,79,3959.91,9,4,3,4,3.5,Mid-Value Active
2,668,45,1954.34,1,5,2,3,2.5,Low-Value Active
3,640,45,2594.30,29,3,2,4,3.0,High-Value Lapsing
6,669,236,5702.51,0,5,5,5,5.0,High-Value Active
7,660,51,2865.89,9,4,2,4,3.0,Mid-Value Active
...,...,...,...,...,...,...,...,...,...
2496,662,60,4105.29,7,5,3,4,3.5,Mid-Value Active
2497,665,216,6874.67,4,5,5,5,5.0,High-Value Active
2498,666,159,2511.39,3,5,5,3,4.0,High-Value Active


In [15]:
# List of households arriving between 670–711
retained_households = tx[tx.day > 669].household_key.unique()
rfm_active["churn"] = np.where(rfm_active.index.isin(retained_households), 0, 1)

In [16]:
rfm_active.churn.mean()

np.float64(0.0807799442896936)

In [17]:
rfm_active

,last_day,frequency,monetary,recency,r_score,f_score,m_score,value_score,segments,churn
household_key,,,,,,,,,,
1,660,79,3959.91,9,4,3,4,3.5,Mid-Value Active,0
2,668,45,1954.34,1,5,2,3,2.5,Low-Value Active,1
3,640,45,2594.30,29,3,2,4,3.0,High-Value Lapsing,0
6,669,236,5702.51,0,5,5,5,5.0,High-Value Active,0
7,660,51,2865.89,9,4,2,4,3.0,Mid-Value Active,0
...,...,...,...,...,...,...,...,...,...,...
2496,662,60,4105.29,7,5,3,4,3.5,Mid-Value Active,0
2497,665,216,6874.67,4,5,5,5,5.0,High-Value Active,0
2498,666,159,2511.39,3,5,5,3,4.0,High-Value Active,0


174 of 2,154 active households (8.1%) made no purchase in days 670–711 and are labelled as churned. The classes are imbalanced, so accuracy alone would be misleading.

### phase 3.3 - feature engineering